In [ ]:
include( "../model.jl" )
include( "../cluster.jl" )
include( "../prob.jl" )
include( "../visual.jl" )

include( "../0-params/rho-list.jl" )
include( "../0-params/nos-list.jl" )

using Optim

home, base = expfolder( "system" )

colormap = cgrad( :ice; rev=true )

In [ ]:
# Hyper parameters.
N = 1024;  s = 0.251

# Create folder and extract critical noise.
folderlist = [findfolder( N, ρ, s; base=base, crit=true ) for ρ ∈ ρlist]

# Extract critical noise values.
ηclist = first.( readdlm.( folderlist.*"../order-0.5.txt" ) );

In [ ]:
# Initialize progress meter.
meter = Progress( Nρ, "Importing data."; dt=1.0 )

# Initialize data variables.
Fdata = Vector{Matrix{defInt}}( undef, Nρ )
θdata = Vector{Matrix{defFloat}}( undef, Nρ )

# Import cluster dynamical data.
for i ∈ 1:Nρ
    # Check that the folder path is valid.
    check1 = ispath( folderlist[i]*"frequency-map.txt" )
    check1 = ispath( folderlist[i]*"optimal-rates.txt" )

    # Unpack the frequency data and the optimal rates.
    Fdata[i] = check1 ? round.( defInt, readdlm( folderlist[i]*"frequency-map.txt" ) ) : fill( -1, N, N )
    θdata[i] = check1 ? readdlm( folderlist[i]*"optimal-rates.txt" ) : fill( NaN, N, 3 )

    # Update progress meter.
    next!( meter )
end

In [ ]:
# Focal parameters to create heatmap.
n = 32       # Subset to consider.
i = 21   # Density.
println( ρlist[i] )

# Extract change frequency and build probability map.
ΔFlist = sum( Fdata[i][1:n,1:n]; dims=2 )
Ω = Fdata[i][1:n,1:n]./ΔFlist

# Extract closest cluster mass distribution.
ΔI = round.( defInt, vec( readdlm( folderlist[i]*"cluster-frequency.txt" ) ) )
Δi = ΔI./sum( ΔI )

# Extrat power-law exponent.
λ = 10.0^optimize( createzipfloss( N, ΔI ), [log10( 1.5 )] ).minimizer[1]
println( λ )

In [ ]:
# Subset of cluster population to consider.
n̂ = 128
x = 1:n̂;  y = 1:n̂
z = Fdata[i][1:n̂,1:n̂]./sum( Fdata[i][1:n̂,1:n̂]; dims=2 )
logz = log10.( z )

# Plot the probability map.
plt = plot(; size=(300,300), legend=false )
plot!( plt; right_margin=0pt )

zmin = floor( minimum( logz[isfinite.( logz )] ) )
zmax = ceil(  maximum( logz[isfinite.( logz )] ) )
heatmap!( plt, x, y, logz';
    color=cgrad( colormap1; rev=false ), colorbar=false, clims=(zmin,zmax) )
contour!( plt, x, y, logz'; color=:black, lw=1 )

plot!( plt; xlims=(0,n̂), xticks=0:n̂/4:n̂ )
plot!( plt; ylims=(0,n̂), yticks=0:n̂/4:n̂ )
plot!( plt; xlabel="original cluster size, "*L"i", ylabel="new cluster size, "*L"j" )
plot!( plt; title="(b)", titleposition=:left )

In [ ]:
# Subset of cluster population to consider.
n̂ = 128;  x = 1:n̂;  y = 1:n̂
z = (Fdata[i][1:n̂,1:n̂]./sum( Fdata[i][1:n̂,1:n̂]; dims=2 ))'

# Plot a single row of the cluster change distribution.
plt = plot(; layout=@layout( [[a{0.35h}; [b{0.95w} c]] [d{0.60w}; e; f]] ), size=(650,450) )
plot!( plt; right_margin=10pt, legend=false )

# Plot the cluster mass distribution.
plot!( plt[1]; legend=:bottomleft )

ιlist = 0.0 .< Δi
scatter!( plt[1], (1:N)[ιlist], Δi[ιlist]; color=:cornflowerblue, markerstrokewidth=0, label="raw data" )
plot!( plt[1], 1:N, zipf.( 1:N; b=N, λ=λ ); color=:black, lw=2, label="power-law" )

ymin = floor( log10( minimum( Δi[ιlist] ) ) )
xticks = round.( defInt, 2.0.^(0:2:log2( N )) )
plot!( plt[1]; xlims=(1,N), xscale=:log10, xticks=(xticks,string.(xticks)) )
plot!( plt[1]; ylims=10.0.^(ymin,0), yscale=:log10 )
plot!( plt[1]; xlabel="cluster size, "*L"i", ylabel="probability, "*L"P(i)" )
plot!( plt[1]; title="(a)", titleposition=:left )

# Plot the probability map.
plot!( plt[2]; right_margin=0pt )

zmin = floor( minimum( logz[isfinite.( logz )] ) )
zmax = ceil(  maximum( logz[isfinite.( logz )] ) )
heatmap!( plt[2], x, y, logz';
    color=cgrad( colormap1; rev=false ), colorbar=false, clims=(zmin,zmax) )
contour!( plt[2], x, y, logz'; color=:black, lw=1 )
plot!( plt[2], [-10,N+10], [-10,N+10]; color=:white, lw=2 )

plot!( plt[2]; xlims=(0,n̂), xticks=0:n̂/4:n̂ )
plot!( plt[2]; ylims=(0,n̂), yticks=0:n̂/4:n̂ )
plot!( plt[2]; xlabel="original cluster size, "*L"i", ylabel="new cluster size, "*L"j" )
plot!( plt[2]; title="(b)", titleposition=:left )

# Plot the colorbar.
m = 1000
vals = reshape( range( zmin,zmax,length=m ),:,1 )
yticks=(0:200:1000,[latexstring( "10^{$(i)}" ) for i ∈ -5:0])
plot!( plt[3]; right_margin=15pt )
heatmap!( plt[3], vals; color=cgrad( colormap1; rev=false ), clims=(zmin,zmax),
    colorbar=false, xtick=false, xticks=false,
    framestyle=:box, mirror=true, yticks=yticks )
plot!( plt[3]; ylabel="probability, "*L"ω_{i,j}" )
# annotate!( plt[3], 0.0, 10^(-2.5), text( "probability, "*L"ω_{i,j}", "Computer Modern", 10, :center ) )

# Plot the row from Ω associated with i=1.
k = 1;  flist = Ω[k,:]./sum( Ω[k,:] )
ιlist = (1:n .< n) .& (0.0 .< flist)
scatter!( plt[4], (1:n)[ιlist], flist[ιlist]; color=:cornflowerblue, markerstrokewidth=0 )

θ = θdata[i][k,:]
println( θ )
plot!( plt[4], 1:n, P.( 1:n; N=n, i=k, λℓ=θ[1], λr=θ[2], κ=θ[3] ); color=:black, lw=2 )

xticks = round.( defInt, 2.0.^(0:round( defInt, log2( n ) )) )
ymin = floor( log10( minimum( flist[ιlist] ) ) )
plot!( plt[4]; xlims=(1,xticks[end]+1), xscale=:log10, xticks=(xticks,string.( xticks )) )
plot!( plt[4]; ylims=10.0.^(ymin,0), yscale=:log10 )
plot!( plt[4]; xlabel="new cluster size, "*L"j", ylabel="probability, "*latexstring( "ω_{$(k),j}" ) )
plot!( plt[4]; title="(c)", titleposition=:left )

# Plot the row from Ω associated with i=n/2.
k = round( defInt, n/2 );  flist = Ω[k,:]
ιlist = (1:n .< n) .& (0.0 .< flist)
scatter!( plt[5], (1:n)[ιlist], flist[ιlist]; color=:cornflowerblue, markerstrokewidth=0 )

θ = θdata[i][k,:]
println( θ )
plot!( plt[5], 1:n, P.( 1:n; N=n, i=k, λℓ=θ[1], λr=θ[2], κ=θ[3] ); color=:black, lw=2 )

ymin = floor( log10( minimum( flist[ιlist] ) ) )
plot!( plt[5]; xlims=(1,xticks[end]+1), xscale=:log10, xticks=(xticks,string.( xticks )) )
plot!( plt[5]; ylims=10.0.^(ymin,0), yscale=:log10 )
plot!( plt[5]; xlabel="new cluster size, "*L"j", ylabel="probability, "*latexstring( "ω_{$(k),j}" ) )
plot!( plt[5]; title="(d)", titleposition=:left )

# Plot the row from Ω associated with i=n.
k = n;  flist = Ω[k,:]
ιlist = (1:n .< n) .& (0.0 .< flist)
scatter!( plt[6], (1:n)[ιlist], flist[ιlist]; color=:cornflowerblue, markerstrokewidth=0 )

θ = θdata[i][k,:]
println( θ )
plot!( plt[6], 1:n, P.( 1:n; N=n, i=k, λℓ=θ[1], λr=θ[2], κ=θ[3] ); color=:black, lw=2 )

ymin = floor( log10( minimum( flist[ιlist] ) ) ) - 1
plot!( plt[6]; xlims=(1,xticks[end]+1), xscale=:log10, xticks=(xticks,string.( xticks )) )
plot!( plt[6]; ylims=10.0.^(ymin,0), yscale=:log10 )
plot!( plt[6]; xlabel="new cluster size, "*L"j", ylabel="probability, "*latexstring( "ω_{$(k),j}" ) )
plot!( plt[6]; title="(e)", titleposition=:left )

plot!( plt; framestyle=:box )

# saveplot( plt, figurefolder*"next-step-distribution_N-$(N)_n-$(n)"; dpi=600 );